# `gain_skeletons` demonstrator

`gain_skeletons` builds mock `xarray` datasets that scaffold radio
interferometric calibration ("gain") solutions, writes them to zarr, and
reads them back. **It is a demonstrator: every value in every dataset below
is randomly generated, and nothing in this package computes or applies
calibration.**

It ships a catalogue of eleven calibration types, split into the
direction-independent ones (phenomenological gain, antenna gain,
tropospheric gain, opacity, bandpass, leakage, delay, antenna positions,
fringe fit) and the direction-dependent ones (direction-dependent
phenomenological gain, ionosphere). Keys are spelled out rather than
abbreviated to the single letters convention assigns some of them. The
catalogue is illustrative rather than exhaustive: it exists to cover the
range of coordinate shapes these datasets take, and the last section shows
that a type it does not carry needs no change to the package. This notebook
walks through the catalogue using only the package's public API, so every
dataset shown here comes from `gain_skeletons` itself; the notebook defines
no schema of its own.

## Imports and the catalogue

`gain_skeletons` is imported under the conventional short alias `gs`.
`list_cal_types` returns the registry keys, direction-independent types
first.

In [1]:
import xarray as xr

import gain_skeletons as gs

gs.list_cal_types()

('phenomenological_gain',
 'antenna_gain',
 'tropospheric_gain',
 'opacity',
 'bandpass',
 'leakage',
 'delay',
 'antenna_positions',
 'fringe_fit',
 'dd_phenomenological_gain',
 'ionosphere')

## Coordinate factories on their own

Each axis the package uses has a standalone factory function that returns a
one-dimensional `xarray.DataArray`. They are usable independently of any
dataset, and their default ranges (a MeerKAT L-band frequency span, a
fixed time origin) are overridable via keyword arguments.

In [2]:
print(gs.time_coord(3, start=0.0, interval=8.0).values)
print(gs.frequency_coord(5).values)
print(gs.frequency_coord(5, start=1.0e9, end=2.0e9).values)
print(gs.antenna_name_coord(4).values)
gs.frequency_coord(4)

[ 0.  8. 16.]
[8.560e+08 1.070e+09 1.284e+09 1.498e+09 1.712e+09]
[1.00e+09 1.25e+09 1.50e+09 1.75e+09 2.00e+09]
['m000' 'm001' 'm002' 'm003']


<xarray.DataArray 'frequency' (frequency: 4)> Size: 32B
array([8.56000000e+08, 1.14133333e+09, 1.42666667e+09, 1.71200000e+09])
Dimensions without coordinates: frequency
Attributes:
    type:                 spectral_coord
    units:                Hz
    observer:             topo
    reference_frequency:  856000000.0
    channel_width:        285333333.3333333

## The simplest case, `antenna_gain`

The standard electronic gain is a complex, on-diagonal-only gain with one
solution for the whole band. Read the dataset repr below against that
description: four axes, a single complex `GAIN` array, units of `rel`, and a
frequency axis of length one — present, but unresolved, which is not the
same thing as absent.

In [3]:
gs.make_gain_xds("antenna_gain")

<xarray.Dataset> Size: 720B
Dimensions:         (time: 4, antenna_name: 8, frequency: 1, receptor_label: 2)
Coordinates:
  * time            (time) float64 32B 1.7e+09 1.7e+09 1.7e+09 1.7e+09
  * antenna_name    (antenna_name) <U4 128B 'm000' 'm001' ... 'm006' 'm007'
  * frequency       (frequency) float64 8B 8.56e+08
  * receptor_label  (receptor_label) <U1 8B 'X' 'Y'
Data variables:
    GAIN            (time, antenna_name, frequency, receptor_label) complex64 512B ...
    FLAG            (time, antenna_name, frequency) bool 32B False ... False
Attributes:
    cal_type:             antenna_gain
    direction_dependent:  False
    jones_structure:      diagonal
    description:          Standard electronic gain, on-diagonal only, one sol...

## `bandpass` against `antenna_gain`: channel-resolved against single-channel

`bandpass` has the same axis list as `antenna_gain`, but is resolved per
channel rather than carrying one solution for the whole band. Both are
on-diagonal-only complex gains, so their `GAIN` arrays share the same
dimensions; only the frequency axis's extent differs.

In [4]:
bandpass = gs.make_gain_xds("bandpass", n_frequency=64)
antenna_gain = gs.make_gain_xds("antenna_gain")
print("bandpass dims    :", dict(bandpass.sizes))
print("antenna_gain dims:", dict(antenna_gain.sizes))
print("same axes:", bandpass.GAIN.dims == antenna_gain.GAIN.dims)
bandpass

bandpass dims    : {'time': 4, 'antenna_name': 8, 'frequency': 64, 'receptor_label': 2}
antenna_gain dims: {'time': 4, 'antenna_name': 8, 'frequency': 1, 'receptor_label': 2}
same axes: True


<xarray.Dataset> Size: 35kB
Dimensions:         (time: 4, antenna_name: 8, frequency: 64, receptor_label: 2)
Coordinates:
  * time            (time) float64 32B 1.7e+09 1.7e+09 1.7e+09 1.7e+09
  * antenna_name    (antenna_name) <U4 128B 'm000' 'm001' ... 'm006' 'm007'
  * frequency       (frequency) float64 512B 8.56e+08 8.696e+08 ... 1.712e+09
  * receptor_label  (receptor_label) <U1 8B 'X' 'Y'
Data variables:
    GAIN            (time, antenna_name, frequency, receptor_label) complex64 33kB ...
    FLAG            (time, antenna_name, frequency) bool 2kB False ... False
Attributes:
    cal_type:             bandpass
    direction_dependent:  False
    jones_structure:      diagonal
    description:          Standard bandpass, on-diagonal only, resolved in fr...

## `phenomenological_gain`: the generic case

A `phenomenological_gain` is a generic representation of any 2x2 Jones matrix. These are useful
as an interchange format as they don't need to be evaluated. These currently use
`gain_X` and `gain_Y` as their parameter labels, but this is still up for debate. This term
type has a sibling called `dd_phenomenological_gain` which includes a direction axis. In
priniciple, we could consolidate these types and either require that the direction axis is 
always present or, alternatively, provide an attribute which indicates whether the term
is direction dependent.

In [5]:
complex_gain = gs.make_gain_xds("phenomenological_gain")
complex_gain

<xarray.Dataset> Size: 68kB
Dimensions:          (time: 4, antenna_name: 8, frequency: 64,
                      receptor_label: 2, parameter_label: 2)
Coordinates:
  * time             (time) float64 32B 1.7e+09 1.7e+09 1.7e+09 1.7e+09
  * antenna_name     (antenna_name) <U4 128B 'm000' 'm001' ... 'm006' 'm007'
  * frequency        (frequency) float64 512B 8.56e+08 8.696e+08 ... 1.712e+09
  * receptor_label   (receptor_label) <U1 8B 'X' 'Y'
  * parameter_label  (parameter_label) <U6 48B 'gain_X' 'gain_Y'
Data variables:
    GAIN             (time, antenna_name, frequency, receptor_label, parameter_label) complex64 66kB ...
    FLAG             (time, antenna_name, frequency) bool 2kB False ... False
Attributes:
    cal_type:             phenomenological_gain
    direction_dependent:  False
    jones_structure:      full
    description:          General Jones term, describing the response without...

## `antenna_positions`: axes genuinely absent, and meaningful parameter labels

`antenna_positions` is neither frequency- nor polarisation-dependent, so it
carries no `frequency` and no `receptor_label` axis whatsoever. An absent
axis is materially different from an axis that exists with length one, such
as `antenna_gain`'s frequency axis above: the first says the quantity has no
such dependence, the second says it has one solution across that dependence.
Its three same-unit components — an antenna position offset in each of X, Y
and Z — instead occupy a `parameter_label` axis.

In [6]:
antenna_positions = gs.make_gain_xds("antenna_positions")
print("axes present:", antenna_positions.ANTENNA_POSITION_OFFSET.dims)
print("frequency absent:", "frequency" not in antenna_positions.dims)
print("parameter labels:", list(antenna_positions.parameter_label.values))
antenna_positions

axes present: ('time', 'antenna_name', 'parameter_label')
frequency absent: True
parameter labels: [np.str_('dX'), np.str_('dY'), np.str_('dZ')]


<xarray.Dataset> Size: 984B
Dimensions:                  (time: 4, antenna_name: 8, parameter_label: 3)
Coordinates:
  * time                     (time) float64 32B 1.7e+09 1.7e+09 1.7e+09 1.7e+09
  * antenna_name             (antenna_name) <U4 128B 'm000' 'm001' ... 'm007'
  * parameter_label          (parameter_label) <U2 24B 'dX' 'dY' 'dZ'
Data variables:
    ANTENNA_POSITION_OFFSET  (time, antenna_name, parameter_label) float64 768B ...
    FLAG                     (time, antenna_name) bool 32B False False ... False
Attributes:
    cal_type:             antenna_positions
    direction_dependent:  False
    description:          Antenna position correction. Three same-unit compon...

## `ionosphere`: the direction axis

`ionosphere` is one of the two direction-dependent types. `direction` is an
integer index into a direction list held elsewhere (a facet within one MSv4
field of view), not a sky position itself. The axis appears only for
calibration types with genuine direction dependence; none of the
direction-independent types carry it.

In [7]:
gs.make_gain_xds("ionosphere", n_direction=4)

<xarray.Dataset> Size: 1kB
Dimensions:       (direction: 4, time: 4, antenna_name: 8)
Coordinates:
  * direction     (direction) int64 32B 0 1 2 3
  * time          (time) float64 32B 1.7e+09 1.7e+09 1.7e+09 1.7e+09
  * antenna_name  (antenna_name) <U4 128B 'm000' 'm001' 'm002' ... 'm006' 'm007'
Data variables:
    TEC           (direction, time, antenna_name) float64 1kB 0.6287 ... 1.292
    FLAG          (direction, time, antenna_name) bool 128B False ... False
Attributes:
    cal_type:             ionosphere
    direction_dependent:  True
    description:          Ionospheric total electron content. Direction-depen...

## `delay`: a parameterised type

`delay` stores the parameters of a phase ramp across frequency — an offset
in degrees and a slope in seconds — rather than the ramp itself sampled
channel by channel. Two things follow. Its `frequency` axis is present but
single-channel, since one ramp is solved for the whole band; and because it
holds two quantities rather than one, it is the first type in this notebook
to produce more than one data array.

The two quantities have different units, and each array simply carries its
own. Both are polarised, so both arrays have the same shape — the case where
they do not is fringe fit, next.

In [8]:
delay = gs.make_gain_xds("delay")
print("frequency extent:", delay.sizes["frequency"])
for name, array in delay.data_vars.items():
    if name != "FLAG":
        print(f"  {name:<8} {str(array.dims):<52} units={array.attrs['units']}")
print("  FLAG    ", delay.FLAG.dims)
delay

frequency extent: 1
  PHASE    ('time', 'antenna_name', 'frequency', 'receptor_label') units=deg
  DELAY    ('time', 'antenna_name', 'frequency', 'receptor_label') units=s
  FLAG     ('time', 'antenna_name', 'frequency')


<xarray.Dataset> Size: 1kB
Dimensions:         (time: 4, antenna_name: 8, frequency: 1, receptor_label: 2)
Coordinates:
  * time            (time) float64 32B 1.7e+09 1.7e+09 1.7e+09 1.7e+09
  * antenna_name    (antenna_name) <U4 128B 'm000' 'm001' ... 'm006' 'm007'
  * frequency       (frequency) float64 8B 8.56e+08
  * receptor_label  (receptor_label) <U1 8B 'X' 'Y'
Data variables:
    PHASE           (time, antenna_name, frequency, receptor_label) float64 512B ...
    DELAY           (time, antenna_name, frequency, receptor_label) float64 512B ...
    FLAG            (time, antenna_name, frequency) bool 32B False ... False
Attributes:
    cal_type:             delay
    direction_dependent:  False
    description:          Delay. A phase offset and a slope in seconds per re...

## Fringe fit: four quantities from one solve

Fringe fit is the larger of the catalogue's two multi-parameter types —
`PHASE`, `DELAY`, `RATE` and `DISP_DELAY`, all from a single solve — and the
only one whose quantities disagree about whether they are polarised.
`DISP_DELAY` carries no `receptor_label` axis; the other three do.

Each quantity is its own data array, named for it, with exactly the axes it
needs and a scalar `units` attribute. So `DISP_DELAY`'s array is genuinely
one axis smaller than its siblings' rather than padded out over receptors it
does not have. One solve is still one dataset, carrying one `FLAG`.

Note what the dataset does *not* carry: a `parameter_label` axis. Each
array's name already says which quantity it holds, so an axis of length one
restating that would say nothing. Where the axis distinguishes components
*within* one quantity — `antenna_positions`' `dX`, `dY` and `dZ` above — it
is present. That is its only job.

In [9]:
fringe_fit = gs.make_gain_xds("fringe_fit")
for name, array in fringe_fit.data_vars.items():
    if name != "FLAG":
        print(f"  {name:<11} {str(array.dims):<56} units={array.attrs['units']}")
print(f"  {'FLAG':<11} {fringe_fit.FLAG.dims}")
print("\nparameter_label axis present:", "parameter_label" in fringe_fit.dims)

  PHASE       ('time', 'antenna_name', 'frequency', 'receptor_label')  units=deg
  DELAY       ('time', 'antenna_name', 'frequency', 'receptor_label')  units=s
  RATE        ('time', 'antenna_name', 'frequency', 'receptor_label')  units=s/s
  DISP_DELAY  ('time', 'antenna_name', 'frequency')                    units=s
  FLAG        ('time', 'antenna_name', 'frequency')

parameter_label axis present: False


In [10]:
fringe_fit

<xarray.Dataset> Size: 2kB
Dimensions:         (time: 4, antenna_name: 8, frequency: 1, receptor_label: 2)
Coordinates:
  * time            (time) float64 32B 1.7e+09 1.7e+09 1.7e+09 1.7e+09
  * antenna_name    (antenna_name) <U4 128B 'm000' 'm001' ... 'm006' 'm007'
  * frequency       (frequency) float64 8B 8.56e+08
  * receptor_label  (receptor_label) <U1 8B 'X' 'Y'
Data variables:
    PHASE           (time, antenna_name, frequency, receptor_label) float64 512B ...
    DELAY           (time, antenna_name, frequency, receptor_label) float64 512B ...
    RATE            (time, antenna_name, frequency, receptor_label) float64 512B ...
    DISP_DELAY      (time, antenna_name, frequency) float64 256B 1.087e-09 .....
    FLAG            (time, antenna_name, frequency) bool 32B False ... False
Attributes:
    cal_type:             fringe_fit
    direction_dependent:  False
    description:          Fringe fit. Four quantities with differing units, p...

## Units are a scalar attribute, always

One array per quantity means one unit per array, so `units` is a plain
string attribute on every data array with no exceptions. This is the main
reason the layout is the way it is.

An earlier version of this package also offered a *consolidated* layout, in
which all of a type's quantities shared one array indexed by
`parameter_label`. That layout could not keep units in a scalar attribute:
four differently-united quantities in one array make any single `units`
string a false claim. The two ways out were both worse. A `parameter_units`
coordinate worked, but meant a reader had to check either an attribute or a
coordinate depending on the data. A `parameter_label -> unit` mapping in
attrs read nicely but does not subset — after
`.sel(parameter_label="DELAY")` the mapping still described all four labels,
because attributes are inert while coordinates are not.

Units that describe the array rather than positions along an axis avoid the
question entirely, and survive any selection for free.

In [11]:
print("units per array:")
for name, array in fringe_fit.data_vars.items():
    if name != "FLAG":
        print(f"  {name:<11} {array.attrs['units']}")

subset = fringe_fit.isel(time=slice(0, 2), frequency=0)
print("\nafter .isel(time=slice(0, 2), frequency=0):")
print("  DELAY units:", subset.DELAY.attrs["units"])
print("  RATE  units:", subset.RATE.attrs["units"])
print("  parameter_units coordinate anywhere:", "parameter_units" in fringe_fit.coords)

units per array:
  PHASE       deg
  DELAY       s
  RATE        s/s
  DISP_DELAY  s

after .isel(time=slice(0, 2), frequency=0):
  DELAY units: s
  RATE  units: s/s
  parameter_units coordinate anywhere: False


## Nothing is broadcast, and one flag covers the solve

Because each array keeps only the axes its quantity is defined over, the
unpolarised `DISP_DELAY` is smaller than its polarised siblings rather than
carrying repeated copies of itself across `receptor_label`.

The `FLAG` spans every axis some parameter uses, less the component axes, so
it covers `DISP_DELAY` even though that array has one axis fewer. The trade
this layout accepts is locality: the four quantities live in four arrays,
which chunk and compress independently once written, rather than in one
array a reader can slice across.

In [12]:
print("PHASE dims     :", fringe_fit.PHASE.dims)
print("DISP_DELAY dims:", fringe_fit.DISP_DELAY.dims)
print("receptor axis on DISP_DELAY:", "receptor_label" in fringe_fit.DISP_DELAY.dims)

# The flag spans the union of every parameter's axes, less the component axes.
union = {
    dim for name, array in fringe_fit.data_vars.items() if name != "FLAG" for dim in array.dims
}
expected = tuple(
    dim
    for dim in fringe_fit.PHASE.dims
    if dim in union and dim not in ("receptor_label", "parameter_label")
)
print("\nFLAG dims      :", fringe_fit.FLAG.dims)
print("union of parameter axes, less components:", expected)

PHASE dims     : ('time', 'antenna_name', 'frequency', 'receptor_label')
DISP_DELAY dims: ('time', 'antenna_name', 'frequency')
receptor axis on DISP_DELAY: False

FLAG dims      : ('time', 'antenna_name', 'frequency')
union of parameter axes, less components: ('time', 'antenna_name', 'frequency')


## Flagging

Every dataset carries exactly one boolean `FLAG`, however many arrays it
holds, and it is deliberately coarser than the parameter arrays it
describes. `FLAG` never carries `parameter_label` or `receptor_label`,
because those two index the components of a single solution rather than
distinct solutions: the components one quantity is made of, and the
receptors solved together. A solution with one untrustworthy component is
not one whose remaining components can be relied on.

`time`, `antenna_name`, `frequency` and `direction` do index genuinely
separate solutions, so `FLAG` keeps them. A direction-dependent type
therefore carries a flag one axis wider than a direction-independent one.

In [13]:
for key in ("bandpass", "fringe_fit", "antenna_positions", "dd_phenomenological_gain"):
    xds = gs.make_gain_xds(key)
    print(f"{key}:")
    for name, array in xds.data_vars.items():
        print(f"  {name:<24} {array.dims}")

bandpass:
  GAIN                     ('time', 'antenna_name', 'frequency', 'receptor_label')
  FLAG                     ('time', 'antenna_name', 'frequency')
fringe_fit:
  PHASE                    ('time', 'antenna_name', 'frequency', 'receptor_label')
  DELAY                    ('time', 'antenna_name', 'frequency', 'receptor_label')
  RATE                     ('time', 'antenna_name', 'frequency', 'receptor_label')
  DISP_DELAY               ('time', 'antenna_name', 'frequency')
  FLAG                     ('time', 'antenna_name', 'frequency')
antenna_positions:
  ANTENNA_POSITION_OFFSET  ('time', 'antenna_name', 'parameter_label')
  FLAG                     ('time', 'antenna_name')
dd_phenomenological_gain:
  GAIN                     ('direction', 'time', 'antenna_name', 'frequency', 'receptor_label', 'parameter_label')
  FLAG                     ('direction', 'time', 'antenna_name', 'frequency')


## Round-trip to zarr and the on-disk layout

Writing uses `consolidated=False` on both write and read: zarr format 3 does
not specify consolidated metadata, so omitting the flag draws a
`ZarrUserWarning` on write and a `RuntimeWarning` on read (because it hunts
for metadata that was never written). Passing `consolidated=True` on read
does not warn — it raises `ValueError` outright, since it demands metadata
that is not there.

Note that `consolidated` here is zarr's metadata flag and has nothing to do
with the removed dataset layout of the same name.

A zarr store holds one directory per array, data variables and coordinates
alike, so the listing below shows fringe fit's four quantities as four
directories that chunk and compress independently. Everything lives in a
`tempfile.TemporaryDirectory`, so nothing is left on disk afterwards.

In [14]:
import tempfile
from pathlib import Path

with tempfile.TemporaryDirectory() as tmp:
    root = Path(tmp)
    store = root / "fringe_fit.zarr"

    fringe_fit.to_zarr(store, consolidated=False)

    print("arrays in the store:")
    for entry in sorted(path.name for path in store.iterdir() if path.is_dir()):
        kind = "data" if entry in fringe_fit.data_vars else "coord"
        print(f"  {entry:<18} ({kind})")

    reread = xr.open_dataset(store, engine="zarr", consolidated=False).load()
    print("\nround-trip identical:", reread.identical(fringe_fit))
    print("units survived      :", reread.RATE.attrs["units"])

arrays in the store:
  DELAY              (data)
  DISP_DELAY         (data)
  FLAG               (data)
  PHASE              (data)
  RATE               (data)
  antenna_name       (coord)
  frequency          (coord)
  receptor_label     (coord)
  time               (coord)

round-trip identical: True
units survived      : s/s


## The escape hatch

The eleven registered calibration types are a convenience, not a limitation.
`CalSpec` and `ParamSpec` are public, so a calibration type the registry
does not carry — here, an antenna pointing offset — can be hand-written and
passed to `make_gain_xds` exactly like a registry name. The registry is a
catalogue of examples, not the whole of what the package can express.

In [15]:
pointing = gs.CalSpec(
    name="pointing_offset",
    parameters=(
        gs.ParamSpec(
            name="POINTING_OFFSET",
            units="rad",
            axes=("time", "antenna_name", "parameter_label"),
            dtype="float64",
            labels=("dAZ", "dEL"),
            scale=1.0e-4,
        ),
    ),
    default_sizes={"time": 4, "antenna_name": 8},
    description="Antenna pointing correction; not in the registry.",
)

gs.make_gain_xds(pointing)

<xarray.Dataset> Size: 728B
Dimensions:          (time: 4, antenna_name: 8, parameter_label: 2)
Coordinates:
  * time             (time) float64 32B 1.7e+09 1.7e+09 1.7e+09 1.7e+09
  * antenna_name     (antenna_name) <U4 128B 'm000' 'm001' ... 'm006' 'm007'
  * parameter_label  (parameter_label) <U3 24B 'dAZ' 'dEL'
Data variables:
    POINTING_OFFSET  (time, antenna_name, parameter_label) float64 512B 1.257...
    FLAG             (time, antenna_name) bool 32B False False ... False False
Attributes:
    cal_type:             pointing_offset
    direction_dependent:  False
    description:          Antenna pointing correction; not in the registry.